In [1]:
import pandas as pd
import numpy as np

In [2]:
#import data from /Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/ProcessedObservedData.csv
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\data\processed\ProcessedObservedData.csv')
#filter by opDiv CDC
data = data[data['OpDiv'] == 'VA']
data.drop(columns=['curr_date', 'OpDiv'], inplace=True)
# Rename the index to 'date' since 'obs_date' is now the index
data.rename(columns={'obs_date': 'date'}, inplace=True)
data['date'] = pd.to_datetime(data['date'])
data.reset_index(drop=True, inplace=True)

data.tail(10)

,indicator,API_UserName,date,observations
6877,104.152.52.148,74198686107399967946,2025-06-09,2
6878,64.32.17.130,74198686107399967946,2025-06-09,3
6879,213.209.129.21,74198686107399967946,2025-06-09,3
6880,216.131.72.131,74198686107399967946,2025-06-09,3
6881,169.150.223.208,74198686107399967946,2025-06-09,2
6882,194.230.148.99,74198686107399967946,2025-06-09,2
6883,192.200.148.241,74198686107399967946,2025-06-09,2
6884,192.200.148.242,74198686107399967946,2025-06-09,2
6885,www.deepseek.com.cdn.cloudflare.net,74198686107399967946,2025-06-09,1
6886,cementexemplifybuddy.com,74198686107399967946,2025-06-09,1


In [3]:
data[data['date'] == '2025-05-30']

,indicator,API_UserName,date,observations
4758,103.133.107.28,74198686107399967946,2025-05-30,255
4759,103.156.92.159,74198686107399967946,2025-05-30,104
4760,103.194.47.162,74198686107399967946,2025-05-30,85
4761,103.207.37.51,74198686107399967946,2025-05-30,144
4762,104.128.161.233,74198686107399967946,2025-05-30,17
...,...,...,...,...
4928,uploads-ssl.webflow.com,74198686107399967946,2025-05-30,100
4929,uploads.strikinglycdn.com,74198686107399967946,2025-05-30,4
4930,vtext.com,74198686107399967946,2025-05-30,54
4931,www.deepseek.com.cdn.cloudflare.net,74198686107399967946,2025-05-30,4


In [4]:
data['date'] = pd.to_datetime(data['date'])

# Define ranges for combinations
all_users = data['API_UserName'].unique()  # Unique API_UserName
all_indicators = data['indicator'].unique()  # Unique indicators
all_dates = pd.date_range(start='2024-01-01', end=pd.Timestamp.now().normalize(), freq='D')

# Create all combinations
all_combinations = pd.MultiIndex.from_product(
    [all_users, all_dates, all_indicators],
    names=['API_UserName', 'date', 'indicator']
).to_frame(index=False)

# Merge with the existing data
merged = all_combinations.merge(data, how='left', on=['API_UserName', 'date', 'indicator'])

# Fill missing values in 'observations' with 0 (not seen)
merged['observations'] = merged['observations'].fillna(0).astype(int)

# Display the first few rows of the merged dataset

# Convert the 'date' column to datetime format
merged['date'] = pd.to_datetime(merged['date'])

# Extract day of the week (0=Monday, 6=Sunday)
merged['dayofweek'] = merged['date'].dt.dayofweek

# Determine if the day is a weekend (Saturday=5, Sunday=6)
merged['is_weekend'] = merged['dayofweek'].isin([5, 6])

# Extract additional features if needed
merged['day'] = merged['date'].dt.day
merged['month'] = merged['date'].dt.month

merged['seen'] = (merged['observations'] > 0).astype(int)
merged.head(10)
merged.to_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv', index=False)

In [5]:
merged[merged['date'] == '2025-06-03']

,API_UserName,date,indicator,observations,dayofweek,is_weekend,day,month,seen
310362,74198686107399967946,2025-06-03,102.90.61.13,0,1,False,3,6,0
310363,74198686107399967946,2025-06-03,102.91.94.193,0,1,False,3,6,0
310364,74198686107399967946,2025-06-03,103.225.136.166,0,1,False,3,6,0
310365,74198686107399967946,2025-06-03,122.187.101.182,0,1,False,3,6,0
310366,74198686107399967946,2025-06-03,123.136.100.101,0,1,False,3,6,0
...,...,...,...,...,...,...,...,...,...
310955,74198686107399967946,2025-06-03,194.110.115.3,0,1,False,3,6,0
310956,74198686107399967946,2025-06-03,194.36.191.196,0,1,False,3,6,0
310957,74198686107399967946,2025-06-03,196.251.73.140,0,1,False,3,6,0
310958,74198686107399967946,2025-06-03,38.240.225.37,0,1,False,3,6,0
